# ___Phylogenetic traversal of ACEs___
---------------------------------------------------

In [1]:
print(R.version$version.string)

[1] "R version 4.5.2 (2025-10-31 ucrt)"


In [1]:
suppressPackageStartupMessages({
    library("ape")
    library("phytools")
    library("corHMM")
    library("diversitree")
})

set.seed(2026-1-26)

In [2]:
STATES <- read.csv("../../data/chapter2/FREDv3subset/finalized_states_395_species.csv", stringsAsFactors = TRUE)[, c("binominal", "state")] # finalized mycorrhizal states
COLLAB_AXIS <- read.csv("../../data/chapter2/FREDv3subset/collab_ord1_species_avgs_SRL_RD.csv", stringsAsFactors = TRUE) # first order species averaged RD and SRL values
MERGED <- merge(x = STATES, y = COLLAB_AXIS, by = "binominal") # merge the two datasets into one, based on the binominal names
stopifnot(nrow(MERGED)==395)

PHYLOGENY <- ape::multi2di(ape::read.tree("../../data/chapter2/uphylomaker/FRED_subset_collab_395sp.tre")) # phylogenetic tree created for the 395 species using U.PhyloMaker
stopifnot(length(PHYLOGENY$tip.label)==395)

# MERGED contains spaces in the binominal names - replace that with underscores; and '/' in mycorrhizal states that need to be removed
data <- data.frame(binominal = gsub(MERGED$binominal, pattern = ' ', replacement = '_'), RD = MERGED$F00679, SRL = MERGED$F00727, myco = gsub(x = MERGED$state, pattern = '/', replacement = ''))
matched_row_indices <- match(PHYLOGENY$tip.label, data$binominal)
stopifnot(all(data$binominal[matched_row_indices] == PHYLOGENY$tip.label))
data <- data[matched_row_indices, ] # reorder the dataset to match the species order in the phylogeny
stopifnot(all(data$binominal == PHYLOGENY$tip.label))
stopifnot(length(unique(data$binominal)) == length(data$binominal))

In [3]:
# FOR CONVENIRNCE
RD <- setNames(object = data$RD, nm = data$binominal)
SRL <- setNames(object = data$SRL, nm = data$binominal)
STATES <- setNames(object = data$myco, nm = data$binominal)

In [4]:
table(STATES) # welp

STATES
   AM AMEcM  AMNM   EcM   ErM    NM 
  300    15     8    65     3     4 

### ___ACE of discrete categorical traits using `ape`, `corHMM`, `phytools` & `diversitree`___
------------------------------------------------------

In [5]:
ape::is.ultrametric(PHYLOGENY) # diversitree requires the phylogeny to be ultrametric

[1] TRUE

In [6]:
# diversitree::make.mkn requires the states to be encodes as numbers
STATES_TO_NUMERIC <- setNames(seq_along(unique(STATES)), unique(STATES)) # numeric encoding of the discrete character states
STATES_NUMERIC <- setNames(unname(STATES_TO_NUMERIC[unname(STATES)]), names(STATES)) # named vector for states with numeric encodings

In [7]:
# diversitree::make.mk2() can only be used with binary state discrete characters
fnlikelihood <- diversitree::make.mkn(tree = PHYLOGENY, states = STATES_NUMERIC, k = length(unique(STATES)))

In [8]:
fnlikelihood

Mk(n) likelihood function:
  * Parameter vector takes 30 elements:
     - q12, q13, q14, q15, q16, q21, q23, q24, q25, q26, q31, q32, q34,
       q35, q36, q41, q42, q43, q45, q46, q51, q52, q53, q54, q56, q61,
       q62, q63, q64, q65
  * Function takes arguments (with defaults)
     - pars: Parameter vector
     - root [ROOT.OBS]: Type of root treatment
     - root.p [NULL]: Vector of root state probabilities
     - intermediates [FALSE]: Also return intermediate values?
  * Phylogeny with 395 tips and 394 nodes
     - Taxa: Anaphalis_aureopunctata, Anaphalis_hancockii, ...
  * References:
     - Pagel (1994)
     - Lewis (2001)
R definition:
function (pars, root = ROOT.OBS, root.p = NULL, intermediates = FALSE)

In [40]:
# pars argument of the likelihood function
# for make.mkn, a vector of k(k-1) parameters, in the order q12,q13,...q1k, q21,q23,...,q2k,...qk(k-1),
# corresponding to the off-diagonal elements of the Q matrix in row order. The order of parameters can be seen by running argnames(f)

# in our case k is 6, so we need (6x(6-1)) 30 values???

In [10]:
tm <- Sys.time()
# this is an ARD model!
dtmod <- diversitree::find.mle(func = fnlikelihood, x.init = rep(x = 1, times = 30), # x.init and any extra args will get passed to the likelihood function
                      root = diversitree::ROOT.FLAT)
tm <- Sys.time() - tm

Warning message in mle.search(func2, x.init, control, lower, upper):
"Convergence problems in find.mle (subplex): number of function evaluations exceeds 'maxit'"


In [14]:
dtmod$lnLik

[1] -277.7409

In [57]:
#----------------------------------------------------------------------------------------------------------------------------------------------------------
# choosing the ARD model for mycorrhizal state evolution as it has been demonstrated that different transitions happen at different rates
# and some transitions are prectically irreversible compared to others
#----------------------------------------------------------------------------------------------------------------------------------------------------------

# there are multiple ways to do ACE of discrete characters 
# ace_mystates_rr <- phytools::rerootingMethod(tree = PHYLOGENY, x = STATES, model = "ARD") - phytools::rerootingMethod method should not be used with non-symmetrical rate models of evolution
# read more here - https://blog.phytools.org/2023/06/decommissioning-rerootingmethod-and.html
ace_mystates_hmm <- corHMM::corHMM(phy = PHYLOGENY, data = data[, c("binominal", "myco")], model = "ARD", node.states = "marginal", rate.cat = 1)
ace_mystates_ape <- ape::ace(phy = PHYLOGENY, x = STATES, type = "discrete", method = "ML", model = "ARD", marginal = FALSE) # ape::ace used a non traditional way to compute marginal ACEs, see its documentation for more details
# setting marginal = FALSE actually returns the marginal ACEs, see the documentation of ape::ace for more details
ace_mystates_ancr <- phytools::ancr(phytools::fitMk(tree = PHYLOGENY, x = STATES, model = "ARD"))
ace_mystates_dvt <- diversitree::asr.marginal()

You specified 'fixed.nodes=FALSE' but included a phy object with node labels. These node labels have been removed.


Warning message in corHMM::corHMM(phy = PHYLOGENY, data = data[, c("binominal", :
"Branch lengths of 0 detected. Adding 1e-5 to these branches."


State distribution in data:
States:	1	2	3	4	5	6	
Counts:	300	15	8	65	3	4	
Beginning thorough optimization search -- performing 0 random restarts 
Finished. Inferring ancestral states using marginal reconstruction. 


Warning message in sqrt(diag(solve(h))):
"NaNs produced"


In [58]:
ace_mystates_ape


    Ancestral Character Estimation

Call: ape::ace(x = STATES, phy = PHYLOGENY, type = "discrete", method = "ML", 
    model = "ARD", marginal = FALSE)

    Log-likelihood: -181.3674 

Rate index matrix:
      AM AMEcM AMNM EcM ErM NM
AM     .     6   11  16  21 26
AMEcM  1     .   12  17  22 27
AMNM   2     7    .  18  23 28
EcM    3     8   13   .  24 29
ErM    4     9   14  19   . 30
NM     5    10   15  20  25  .

Parameter estimates:
 rate index estimate std-err
          1   0.1675     NaN
          2   0.2740  0.1254
          3   0.0000     NaN
          4   0.0000     NaN
          5   0.0893     NaN
          6   0.0001  0.0006
          7   0.1518  0.0526
          8   0.0012  0.0028
          9   0.1165     NaN
         10   0.1282  0.0334
         11   0.0028  0.0018
         12   0.0425     NaN
         13   0.0000     NaN
         14   0.1164     NaN
         15   0.0822  0.0977
         16   0.0000  0.0008
         17   0.1545     NaN
         18   0.2093  0.1706
     

In [59]:
ace_mystates_ape$lik.anc

,AM,AMEcM,AMNM,EcM,ErM,NM
,0.1780089,0.1676518788,0.1678401104,1.578816e-01,1.634363e-01,1.651812e-01
Spermatophyta,0.6310923,0.0084811117,0.0046860970,3.366761e-01,1.046941e-02,8.594939e-03
Mesangiospermae,0.9977983,0.0008959747,0.0007030378,4.115483e-05,1.866265e-04,3.749249e-04
mrcaott2ott121,0.9976928,0.0009168643,0.0008248789,5.344119e-06,1.821930e-04,3.778822e-04
eudicotyledons,0.9955824,0.0018530047,0.0015131992,1.288869e-04,3.336534e-04,5.888886e-04
mrcaott2ott969,0.9940670,0.0023076272,0.0023267179,7.443955e-05,4.406896e-04,7.835524e-04
Pentapetalae,0.9924739,0.0039979010,0.0019058349,5.098264e-05,5.261451e-04,1.045281e-03
mrcaott248ott19688,0.9834832,0.0065906228,0.0053868733,6.934968e-04,1.511159e-03,2.334679e-03
mrcaott248ott557,0.9799111,0.0082624305,0.0065630318,3.467028e-04,1.854204e-03,3.062496e-03
mrcaott248ott27233,0.9797556,0.0108912152,0.0042882178,4.049869e-04,1.664066e-03,2.995946e-03


In [60]:
ace_mystates_ancr

Marginal ancestral state estimates:
          AM    AMEcM     AMNM      EcM ErM       NM
396 0.127694 0.002593 0.358596 0.001056   0 0.510062
397 0.541962 0.000657 0.204941 0.001657   0 0.250783
398 0.999895 0.000000 0.000087 0.000000   0 0.000019
399 0.999904 0.000000 0.000081 0.000000   0 0.000015
400 0.999756 0.000000 0.000232 0.000000   0 0.000012
401 0.999835 0.000000 0.000156 0.000000   0 0.000009
...

Log-likelihood = -128.722103 


In [61]:
ace_mystates_ancr$ace

,AM,AMEcM,AMNM,EcM,ErM,NM
396,0.1276937,2.592872e-03,3.585956e-01,1.056257e-03,0,5.100616e-01
397,0.5419618,6.572441e-04,2.049406e-01,1.657275e-03,0,2.507831e-01
398,0.9998947,4.907078e-13,8.655514e-05,1.959951e-08,0,1.867626e-05
399,0.9999042,5.140939e-12,8.077173e-05,7.768918e-09,0,1.497816e-05
400,0.9997561,1.896532e-10,2.315145e-04,2.751930e-08,0,1.240614e-05
401,0.9998345,5.011078e-10,1.559156e-04,6.784768e-08,0,9.492852e-06
402,0.9999795,1.427418e-11,1.530185e-05,4.787841e-09,0,5.155376e-06
403,0.9995313,4.819200e-08,2.876598e-04,6.361608e-08,0,1.808865e-04
404,0.9996743,1.036743e-08,2.074287e-04,2.330845e-07,0,1.180267e-04
405,0.9999722,5.051059e-11,2.100304e-05,4.661238e-08,0,6.766740e-06


In [ ]:
ace_mystates_dvt <- diversitree::

In [ ]:
#--------------------------------
# ACE OF CONTINUOUS TRAITS
#--------------------------------

ace_srl_fastanc <- phytools::fastAnc(tree = PHYLOGENY, x = SRL)

In [ ]:
# WE DO HAVE POLYTOMIES IN OUR PHYLOGENY :(

In [6]:
ace_mystates_hmm # WE DO SEE TRANSITION RATE DIFFERENCES 


Fit
      -lnL      AIC     AICc Rate.cat ntax
 -127.3412 314.6824 319.7923        1  395

Legend
      1       2       3       4       5       6 
   "AM" "AMEcM"  "AMNM"   "EcM"   "ErM"    "NM" 

Rates
             (1,R1)      (2,R1)       (3,R1)       (4,R1)       (5,R1)
(1,R1)           NA 0.000322027 0.0009748366 0.0002015374 6.729752e-05
(2,R1) 1.000336e-09          NA 0.0000000010 0.0174455509 1.000000e-09
(3,R1) 1.691330e-02 0.000000001           NA 0.0000000010 1.000000e-09
(4,R1) 1.299848e-03 0.000000001 0.0000000010           NA 1.000000e-09
(5,R1) 1.000000e-09 0.000000001 0.0000000010 0.0000000010           NA
(6,R1) 1.000000e-09 0.000000001 0.0071676917 0.0000000010 1.000000e-09
             (6,R1)
(1,R1) 0.0000000010
(2,R1) 0.0000000010
(3,R1) 0.0102782964
(4,R1) 0.0000000010
(5,R1) 0.0009739048
(6,R1)           NA

Arrived at a reliable solution 

In [23]:
head(ace_mystates_hmm$states) # probabilities of each internal node of the phylogeny belonging to the given states

"(1,R1)","(2,R1)","(3,R1)","(4,R1)","(5,R1)","(6,R1)"
0.7384076,1.040770e-04,0.0993487387,6.261923e-03,6.054074e-04,1.552723e-01
0.8500365,7.905156e-04,0.0746536180,5.613651e-03,2.862179e-05,6.887709e-02
0.9991832,9.825375e-13,0.0008165587,7.341792e-08,1.816180e-15,1.231038e-07
0.9992081,2.686675e-11,0.0007917985,3.029895e-08,2.461497e-14,8.734257e-08
0.9989132,2.536349e-10,0.0010865789,5.341592e-08,3.150591e-12,2.026010e-07
0.9990792,3.114671e-09,0.0009205367,1.242975e-07,2.426444e-12,1.727226e-07


In [29]:
rowSums(ace_mystates_hmm$states) # probabilities of all states add up to 1.00 :)

[1] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 [38] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 [75] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[112] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[149] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[186] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[223] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[260] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[297] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[334] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[371] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1

In [31]:
# figure out which state is the most probable for all the internal nodes
apply(ace_mystates_hmm$states, MARGIN = 1, FUN = which.max) # MARGIN = 1 means apply the function to each row

[1] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 [38] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 5 5 1 1 1 1 1 1 1 1
 [75] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 1 1 1 1 1 1 1
[112] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 4
[149] 4 4 4 4 4 4 4 4 4 4 4 4 4 4 1 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4
[186] 4 1 1 1 1 1 1 1 1 2 2 2 2 2 2 2 2 2 2 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 4
[223] 4 4 4 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[260] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 3 3 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[297] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
[334] 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 4 4 4 4 4 4 4 4 4 4
[371] 4 4 4 4 4 4 4 4 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 6

In [38]:
PHYLOGENY # our phylogeny has 395 tips ans 394 internal nodes and we reconstructed ancestral traits for the internal nodes


Phylogenetic tree with 395 tips and 394 internal nodes.

Tip labels:
  Anaphalis_aureopunctata, Anaphalis_hancockii, Solidago_decurrens, Doellingeria_scabra, Aster_tataricus, Artemisia_igniaria, ...
Node labels:
  , Spermatophyta, Mesangiospermae, mrcaott2ott121, eudicotyledons, mrcaott2ott969, ...

Rooted; includes branch length(s).

In [39]:
dim(ace_mystates_hmm$states) # got 394 rows and 6 columns

[1] 394   6

In [50]:
# based on the legend of the corHMM model output
STATES_LEGEND <- c("AM", "AMEcM", "AMNM", "EcM", "ErM", "NM")
STATES_LEGEND

[1] "AM"    "AMEcM" "AMNM"  "EcM"   "ErM"   "NM"

[1] "AMNM"  "AM"    "ErM"   "NM"    "AMEcM" "EcM"

In [52]:
# reconstructed ancestral states of the internal nodes, with node numbers as names
STATES_LEGEND[apply(ace_mystates_hmm$states, MARGIN = 1, FUN = which.max)]

[1] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
 [10] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
 [19] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
 [28] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
 [37] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
 [46] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
 [55] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
 [64] "AM"    "ErM"   "ErM"   "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
 [73] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
 [82] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
 [91] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[100] "AM"    "AM"    "AM"    "AMEcM" "AMEcM" "AM"    "AM"    "AM"    "AM"   
[109] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[118] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[127] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[136] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[145] "AM"    "AM"    "AM"    "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"  
[154] "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"  
[163] "AM"    "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"  
[172] "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"  
[181] "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "AM"    "AM"    "AM"   
[190] "AM"    "AM"    "AM"    "AM"    "AM"    "AMEcM" "AMEcM" "AMEcM" "AMEcM"
[199] "AMEcM" "AMEcM" "AMEcM" "AMEcM" "AMEcM" "AMEcM" "AMEcM" "AM"    "AM"   
[208] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[217] "AM"    "AM"    "AM"    "AM"    "AM"    "EcM"   "EcM"   "EcM"   "EcM"  
[226] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[235] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[244] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[253] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[262] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[271] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[280] "AM"    "AMNM"  "AMNM"  "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[289] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[298] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[307] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[316] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[325] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[334] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[343] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[352] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[361] "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"  
[370] "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"   "EcM"  
[379] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "AM"   
[388] "AM"    "AM"    "AM"    "AM"    "AM"    "AM"    "NM"

In [ ]:


# WHAT WE NEED HERE IS A DICTIONARY OF WHICH NODES DESCEND FROM WHICH NODES, SO WE CAN LOOK UP THE STATE TRANSITIONS AND CONTINUOUS TRAIT CHANGES
# http://www.phytools.org/eqg/Exercise_3.2/
# By convention, the tips of the tree are numbered 1 through n for n tips; and the nodes are numbered n + 1 through n + m for m nodes
# the matrix edge contains the beginning and ending node number for all the nodes and tips in the tree.
PHYLOGENY$edge

phyedges <- as.data.frame(PHYLOGENY$edge)
colnames(phyedges) <- c("from", "to")

for (i in 1:nrow(phyedges)) {
    print(ace_srl_fastanc[as.character(phyedges[i, ][, "from"])]) # - ace_srl_fastanc[phyedges[i, ][, "to"]])
}

